<p align="center"><img src="https://raw.githubusercontent.com/arab-future-academy/arab-future-academy/main/logo.png" width="200"></p>

# 🚀 ComfyUI on Lightning.ai — Arabic Future Academy

**هذا النوتبوك مصمم خصيصاً لـ Lightning.ai Studio** — بنفس الكتالوج والحزم الموجودة في نسخة Colab.

### الفروقات عن Colab:
| | Colab | Lightning.ai |
|---|---|---|
| التخزين | مؤقت (`/content`) | **دائم** (Studio disk) |
| GPU | T4 مجاني (حد استخدام) | T4/A10G (~22 ساعة شهرياً) |
| الوصول | Cloudflared tunnel | **Port Viewer** (مدمج) |
| PyTorch | قد تحتاج تثبيت | **مثبّت مسبقاً** |
| Google Drive | مطلوب | غير مطلوب |

### خطوات البدء:
1. أنشئ Studio جديد من <https://lightning.ai/comfy>
2. حوّل لـ **GPU** (T4)
3. شغّل هذه الخلايا بالترتيب

---


# **الإعدادات المركزية**

**نفس إعدادات Colab — نفس الكتالوج المثبّت والمراجع (SHA).**

اختر حزمة `BUNDLE` واحدة في الجلسة (أو استخدم `none` + `EXTRA_FILES`).

**ملاحظة Lightning.ai:** التخزين دائم — الموديلات تبقى بين الجلسات! لا حاجة لإعادة التحميل.


In [ ]:
#@title 0. Central Configuration
#@markdown Same catalog and bundles as the Colab version.

# --- Model selection ---
#@markdown **Model bundle:** curated sets from the pinned manifest.
BUNDLE = "none" #@param ["none", "z-image-turbo-t4", "z-image-turbo", "sd15-classic", "qwen-edit-2511", "qwen-image-2.1-edit", "flux2-klein", "chatterbox-tts", "depth-3d-trellis", "3d-pixal3d-trellis2", "wan-vace-t4", "wan-scail2-t4", "extras", "llm-qwen3-8b-t4", "llm-qwen2.5-7b-t4", "llm-qwen2.5-3b-t4", "llm-qwen2.5-1.5b-t4", "llm_gemma4_text_gen", "llm_gemma4_text_gen_t4", "llm_qwen3_5_text_gen", "llm_qwen3_5_text_gen_t4", "starter-t4", "wan-scail2-full", "wan-vace-full"]
#@markdown **Extra files:** one per line — manifest dest or hf://repo/path/file
EXTRA_FILES = "" #@param {type:"string"}

# --- Paths (Lightning.ai persistent disk) ---
import os
LOCAL_WORKSPACE = "/mnt/user-data/ComfyUI"
if not os.path.isdir("/mnt/user-data"):
    LOCAL_WORKSPACE = "/workspace/ComfyUI"  # fallback

MANIFEST_URL = "https://raw.githubusercontent.com/awayham/comfyui-basics-arabic/main/notebooks/model_manifest.yaml"
HF_XET_HIGH_PERF = False

# --- Info ---
_n_extra = len([l for l in EXTRA_FILES.splitlines() if l.strip() and not l.strip().startswith("#")])
print("Configuration loaded.")
print("  Bundle: {} · Extra files: {}".format(BUNDLE, _n_extra))
print("  Workspace: {}".format(LOCAL_WORKSPACE))
print("  Persistent: {}".format("Yes (/mnt/user-data)" if "/mnt/user-data" in LOCAL_WORKSPACE else "No"))


# **تهيئة النظام**

**في Lightning.ai: PyTorch مثبّت مسبقاً — نحتاج فقط تثبيت ComfyUI والمكتبات.**

### قبل التشغيل:
- تأكد إن الـ **Studio** يعمل على **GPU** (T4)
- من الأعلى: **Studio Environment → Switch to GPU**

### ماذا يحدث:
- فحص GPU + PyTorch
- تثبيت/تحديث ComfyUI
- تثبيت المكتبات الأساسية


In [ ]:
#@title 1. System Initialization
#@markdown Installs ComfyUI and core dependencies. PyTorch is pre-installed on Lightning.ai.

import os, subprocess, sys, shutil

WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")
print("[SYSTEM] Workspace: {}".format(WORKSPACE))

def run(cmd, cwd=None, check=True):
    print("$ {}".format(" ".join(cmd) if isinstance(cmd, list) else cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, text=True)
    if check and r.returncode != 0:
        print("[WARN] Command returned {}".format(r.returncode))

# 1. GPU check
print("\n[GPU] Checking...")
try:
    import torch
    if torch.cuda.is_available():
        print("[OK] GPU: {} · torch {}".format(torch.cuda.get_device_name(0), torch.__version__))
        print("[OK] VRAM: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_mem / 1e9))
    else:
        print("[WARN] CUDA not available. Switch to GPU: Studio Environment > Switch to GPU")
except ImportError:
    print("[WARN] torch not found — installing...")
    run([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/cu124"])

# 2. Disk check
for p in ["/mnt/user-data", "/workspace", "/"]:
    if os.path.exists(p):
        total, used, free = shutil.disk_usage(p)
        print("[OK] Disk free ({}) : {:.1f} GB".format(p, free / (1024**3)))
        break

# 3. Clone / update ComfyUI
if not os.path.exists(os.path.join(WORKSPACE, "main.py")):
    print("\n[COMFYUI] Cloning...")
    os.makedirs(os.path.dirname(WORKSPACE), exist_ok=True)
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", WORKSPACE])
else:
    print("\n[COMFYUI] Updating...")
    run(["git", "pull"], cwd=WORKSPACE)

# 4. Manager
manager_path = os.path.join(WORKSPACE, "custom_nodes", "ComfyUI-Manager")
if not os.path.exists(manager_path):
    print("[MANAGER] Installing...")
    run(["git", "clone", "https://github.com/ltdrdata/ComfyUI-Manager.git", manager_path])
else:
    print("[MANAGER] Verified.")

# 5. Dependencies
print("\n[PYTHON] Installing requirements...")
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=WORKSPACE)
run([sys.executable, "-m", "pip", "install", "-q", "gguf", "piexif", "huggingface_hub", "pyyaml", "requests"])

# 6. Xet off
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HOME"] = os.path.join(WORKSPACE, ".hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print("\n[SYSTEM READY] Initialization complete.")


# **تسجيل الدخول لهافينج فيس**

**مطلوب فقط إذا كنت تحمّل من مستودعات خاصة أو تريد rate limit أعلى.**

احفظ التوكن في: **Studio Settings → Secrets → HF_TOKEN**

بدون توكن: التحميل يعمل لكن بسرعة أقل (rate limit مجهول).


In [ ]:
#@title 1b. Hugging Face Login (optional)
import os

hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("[HF] Logged in.")
else:
    print("[HF] No HF_TOKEN found — using anonymous access (slower rate limits).")
    print("     Add it: Studio Settings > Secrets > HF_TOKEN")


# **تحميل الموديلات**

**نفس كتالوج Colab — تحميل مباشر من Hugging Face بمراجع مثبّتة.**

### ⚡ ميزة Lightning.ai:
الملفات تُحفظ في القرص **الدائم** — لا تُحذف بين الجلسات!

اختر حزمة في `BUNDLE` (خلية الإعدادات) ثم شغّل.


In [ ]:
#@title 2. Model Pull (manifest-driven)
#@markdown Pulls the selected bundle from pinned upstream sources.

import os, sys, shutil, tempfile
import requests, yaml
from huggingface_hub import hf_hub_download, HfApi

BUNDLE = globals().get("BUNDLE", "none")
EXTRA_FILES = globals().get("EXTRA_FILES", "")
MANIFEST_URL = globals().get("MANIFEST_URL", "https://raw.githubusercontent.com/awayham/comfyui-basics-arabic/main/notebooks/model_manifest.yaml")
LOCAL_WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")
MIN_FREE_DISK_GB = 15

# Disable Xet
os.environ["HF_HUB_DISABLE_XET"] = "1"

MODELS_ROOT = os.path.join(LOCAL_WORKSPACE, "models")

# 1) Fetch manifest
print("[MANIFEST] Fetching {}".format(MANIFEST_URL))
head = requests.head(MANIFEST_URL, timeout=20, allow_redirects=True)
head.raise_for_status()
manifest = yaml.safe_load(requests.get(MANIFEST_URL, timeout=30).text)
FILES_BY_DEST = {f["dest"]: f for f in manifest.get("files", [])}
BUNDLES = manifest.get("bundles", {})
print("[MANIFEST] {} files · bundles: {}".format(len(FILES_BY_DEST), ", ".join(BUNDLES)))

# 2) Resolve selection
selected = []
if BUNDLE and BUNDLE != "none":
    if BUNDLE not in BUNDLES:
        raise SystemExit("[PULL] Unknown bundle '{}'. Available: {}".format(BUNDLE, list(BUNDLES)))
    for dest in BUNDLES[BUNDLE]:
        if dest not in FILES_BY_DEST:
            raise SystemExit("[PULL] Bundle '{}' references missing entry: {}".format(BUNDLE, dest))
        selected.append(dict(FILES_BY_DEST[dest]))

for raw in (EXTRA_FILES or "").replace(",", "\n").split("\n"):
    spec = raw.strip()
    if not spec or spec.startswith("#"):
        continue
    if spec in FILES_BY_DEST:
        selected.append(dict(FILES_BY_DEST[spec]))
    elif spec.startswith("hf://"):
        ref = spec[len("hf://"):]
        repo_id, _, remote = ref.partition("/")
        if not remote:
            raise SystemExit("[PULL] Bad hf:// ref: {}".format(spec))
        sha = HfApi().model_info(repo_id).sha
        selected.append({"dest": "extras/" + os.path.basename(remote), "source": "hf://" + repo_id,
                         "remote": remote, "revision": sha, "size_gb": None})
    else:
        raise SystemExit("[PULL] Unrecognized: {}".format(spec))

# de-dupe
deduped, seen = [], set()
for j in selected:
    if j["dest"] in seen:
        continue
    seen.add(j["dest"])
    deduped.append(j)
selected = deduped

if not selected:
    print("[PULL] Nothing selected. Skipping.")
else:
    need_gb = sum(j["size_gb"] for j in selected if j.get("size_gb"))
    free_gb = shutil.disk_usage(LOCAL_WORKSPACE if os.path.exists(LOCAL_WORKSPACE) else "/").free / (1024**3)
    print("[DISK] Need ~{:.1f} GB · free: {:.1f} GB".format(need_gb, free_gb))
    if free_gb - need_gb < MIN_FREE_DISK_GB:
        raise SystemExit("[DISK] Not enough space: need {:.1f} + {} margin, have {:.1f} GB".format(need_gb, MIN_FREE_DISK_GB, free_gb))

    results = []
    for j in selected:
        src, dest = j["source"], j["dest"]
        final_path = os.path.join(MODELS_ROOT, dest)
        if os.path.exists(final_path):
            size = os.path.getsize(final_path)
            expect = j.get("size_gb")
            if expect is None or abs(size - expect * 1e9) / max(size, 1) <= 0.02:
                print("   [SKIP] {} ({:.2f} GB)".format(dest, size / 1e9))
                results.append((dest, "skip", size))
                continue
        if src.startswith(("hf://", "archive://")):
            repo_id = src.split("://", 1)[1]
            remote = j.get("remote", dest)
            rev = j.get("revision") or None
            tag = "@" + rev[:10] if rev else "@main"
            print("   [HF] {} :: {} {}".format(repo_id, remote, tag))
            stage = os.path.join(LOCAL_WORKSPACE, ".pull_tmp")
            os.makedirs(stage, exist_ok=True)
            with tempfile.TemporaryDirectory(dir=stage) as td:
                p = hf_hub_download(repo_id=repo_id, filename=remote, revision=rev,
                                    local_dir=td, repo_type="model")
                os.makedirs(os.path.dirname(final_path) or MODELS_ROOT, exist_ok=True)
                shutil.move(p, final_path)
            size = os.path.getsize(final_path)
            print("       -> {} ({:.2f} GB)".format(dest, size / 1e9))
            results.append((dest, "ok", size))
        elif src.startswith(("url://", "http://", "https://")):
            url = src[len("url://"):] if src.startswith("url://") else src
            print("   [URL] {}".format(url))
            os.makedirs(os.path.dirname(final_path) or MODELS_ROOT, exist_ok=True)
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(final_path, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        f.write(chunk)
            size = os.path.getsize(final_path)
            results.append((dest, "ok", size))
        else:
            print("   [ERR] unknown source: {}".format(src))
            results.append((dest, "err", 0))

    print("\n[PULL REPORT]")
    ok = sum(1 for _, s_, _ in results if s_ in ("ok", "skip"))
    bad = sum(1 for _, s_, _ in results if s_ == "err")
    for d, s_, sz in results:
        print("   {:4}  {:6.2f} GB  {}".format(s_.upper(), sz / 1e9, d))
    print("[PULL] done: {} ok/skipped · {} failed".format(ok, bad))


# **تشغيل ComfyUI**

**في Lightning.ai لا نحتاج cloudflared — نستخدم Port Viewer المدمج.**

### بعد التشغيل:
1. اضغط **+** أعلى اليمين
2. اختر **Web Apps → Port Viewer**
3. أضف port **8188**
4. افتح الرابط

أو استخدم الرابط المباشر: `http://localhost:8188` داخل الـ Studio.


In [ ]:
#@title 3. Start ComfyUI
#@markdown Boots ComfyUI. Use Port Viewer (port 8188) to access the UI.

import subprocess, time, socket, os, sys

WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")
PORT = 8188
LOG = os.path.join(WORKSPACE, "comfyui_boot.log")

def port_open(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        return s.connect_ex(("127.0.0.1", port)) == 0
    finally:
        s.close()

# GPU check
try:
    import torch
    if not torch.cuda.is_available():
        print("[WARN] No CUDA. Switch to GPU: Studio Environment > Switch to GPU")
except ImportError:
    print("[WARN] torch missing — re-run cell 1.")

# Boot
if port_open(PORT):
    print("[SYSTEM] ComfyUI already on :{}".format(PORT))
else:
    os.chdir(WORKSPACE)
    open(LOG, "w").close()
    print("[SYSTEM] Booting ComfyUI...")
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(PORT)]
    proc = subprocess.Popen(cmd, stdout=open(LOG, "wb"), stderr=subprocess.STDOUT)
    print("[SYSTEM] PID={} · log={}".format(proc.pid, LOG))

    pos = 0
    up = False
    for i in range(120):
        time.sleep(1)
        if os.path.exists(LOG):
            with open(LOG, "r", errors="replace") as f:
                f.seek(pos)
                chunk = f.read()
                pos = f.tell()
            if chunk:
                sys.stdout.write(chunk)
                sys.stdout.flush()
        if port_open(PORT):
            up = True
            print("\n[OK] Server UP on :{} ({}s)".format(PORT, i + 1))
            break
        if proc.poll() is not None:
            print("\n[FATAL] Process exited ({})".format(proc.returncode))
            print(open(LOG, errors="replace").read()[-2000:])
            raise SystemExit(1)

    if not up:
        print("\n[FATAL] :{} did not open in 120s".format(PORT))
        print(open(LOG, errors="replace").read()[-2000:])
        raise SystemExit(1)

print("=" * 50)
print("  ComfyUI is RUNNING")
print("  Access: Port Viewer -> port {}".format(PORT))
print("  Or open: http://localhost:8188 in the Studio browser")
print("=" * 50)


# **تثبيت Auto-Start (اختياري)**

**يضيف ComfyUI إلى `on_start.sh` ليبدأ تلقائياً مع كل Studio.**


In [ ]:
#@title 4. Auto-Start Setup (optional)
#@markdown Adds ComfyUI boot to on_start.sh for automatic startup.

import os

WORKSPACE = globals().get("LOCAL_WORKSPACE", "/mnt/user-data/ComfyUI")
ON_START = os.path.expanduser("~/.on_start.sh")

boot_line = "cd {} && nohup python main.py --listen 127.0.0.1 --port 8188 > comfyui_boot.log 2>&1 &".format(WORKSPACE)

if os.path.exists(ON_START):
    content = open(ON_START).read()
    if "main.py" in content:
        print("[AUTO-START] Already configured.")
    else:
        with open(ON_START, "a") as f:
            f.write("\n# ComfyUI auto-start\n" + boot_line + "\n")
        print("[AUTO-START] Added to {}".format(ON_START))
else:
    with open(ON_START, "w") as f:
        f.write("#!/bin/bash\n# ComfyUI auto-start\n" + boot_line + "\n")
    os.chmod(ON_START, 0o755)
    print("[AUTO-START] Created {}".format(ON_START))

print("[AUTO-START] Next Studio boot will auto-start ComfyUI.")
